### 0. 환경 설정 및 초기화

In [8]:
import os
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
import time
import numpy as np
from collections import Counter


# LangChain
from langchain.schema import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

# LangGraph
from typing import TypedDict, List, Dict, Any, Optional, Tuple
from langgraph.graph  import StateGraph, END

from dotenv import load_dotenv
load_dotenv()

True

In [10]:
# LLM 초기화
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 품질 임계값 설정
QUALITY_THRESHOLD = 0.6 # 검색 품질이 60% 이상이어야 함.
MAX_RETRIES = 2 # 최대 2번까지 재검색 시도

## 1. State 정의

- 근데 이렇게 속성을 어떻게 다 정의하는 걸까?

In [11]:
class CosmeticPlanningState(TypedDict):
    # 입력
    original_query: str                 # 사용자가 입력한 원본 쿼리(= 기획안 초안)
    
    # 쿼리 분석
    query_complexity: Dict[str, Any]    # 쿼리가 얼마나 복잡한지
    required_categories: List[str]      # 어떤 종류의 검색이 필요한지

    # Self-query 검색 결과
    extracted_filters: Dict[str, Any]   # 추출된 메타데이터 (브랜드, 가격 등)
    structured_query: str               # 필터를 제외한 순수 검색어
    
    # 쿼리 분해 결과
    decomposed_queries: List[Dict[str, Any]]    # 작은 검색 단위들

    # 검색 결과
    search_results: Dict[str, List[Document]]   # 각 검색 결과
    search_quality_scores: Dict[str, float]     # 각 검색 결과의 품질 점수
    retry_attempts: Dict[str, int]              # 각 검색 결과의 재검색 시도 횟수

    # 추출된 인사이트
    extracted_insights: Dict[str, Any]          # 추출된 인사이트

    # 최종 결과
    draft_proposal: str                         # 초안
    final_proposal: str                         # 최종 기획안
      
    # 메타데이터
    iteration_count: int                        # 전체 반복 횟수  
    feedback: Optional[str]                     # 사용자 피드백
        # 사용자는 개입 안할건데... 알아서 피드백 되도록 해야함. 그러면 기준을 정해야 함!

## 2. Node 정의

### 2-1. 쿼리 복잡도 분석 노드

In [ ]:
def analyze_query_complexity(state: CosmeticPlanningState) -> CosmeticPlanningState:
    """
    쿼리 복잡도를 분석하여 필요한 검색 카테고리 결정
    """
    prompt = ChatPromptTemplate.from_messages([
        (
            "system","""
쿼리를 분석하여 복잡도와 필요한 검색 카테고리를 결정하세요.

가능한 카테고리:
- target_audience: 타겟 고객층 분석 (거의 항상 필요)
- product_benefits: 제품 효능과 기능 (거의 항상 필요)
- differentiation: 차별화 전략
- brand_positioning: 브랜드 포지셔닝 (특정 브랜드 언급시 필요)
- marketing_strategy: 마케팅 전략 (마케팅/홍보 언급시 필요)
- price_analysis: 가격 분석 (가격대ㅔ 언급시 필요)
- ingredient_analysis: 성분 분석 (특성 성분 언급시 필요)
"""
        ),
        ("human", """쿼리: {query}
다음 형식으로 응답하세요:
{{
    "complexity_level": "low" | "medium" | "high",
    "key_aspects": ["식별된 핵심 요소들"],
    "re"

}}
        """)
    ])

# 진짜 최종

In [35]:
import os
from typing import TypedDict, List, Dict, Any, Optional, Annotated, Sequence
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
import functools
from datetime import datetime
import logging
from collections import defaultdict
from pydantic import BaseModel, Field, field_validator

# --- LLM, 저장소, Retriever 관련 라이브러리 ---
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.storage import LocalFileStore, create_kv_docstore
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- 1. Pydantic 모델 정의 (Structured Output) ---
class ProductCategory(BaseModel):
    """제품 카테고리 분류 결과"""
    category: str = Field(
        description="9개 카테고리 중 하나: 스킨케어, 마스크팩, 클렌징, 선케어, 메이크업, 맨즈케어, 헤어케어, 바디케어, 향수/디퓨저"
    )
    confidence: float = Field(
        default=1.0,
        description="분류 신뢰도 (0.0-1.0)"
    )
    
    @field_validator('category')
    def validate_category(cls, v):
        valid_categories = ["스킨케어", "마스크팩", "클렌징", "선케어", "메이크업", "맨즈케어", "헤어케어", "바디케어", "향수/디퓨저"]
        if v not in valid_categories:
            raise ValueError(f"유효하지 않은 카테고리: {v}. 다음 중 하나여야 합니다: {', '.join(valid_categories)}")
        return v

class ResearchQuestions(BaseModel):
    """기획안 검증을 위한 연구 질문"""
    questions: List[str] = Field(
        min_items=5,
        max_items=10,
        description="기획안을 검증하고 발전시키기 위한 핵심 질문들"
    )
    priority_scores: List[float] = Field(
        default_factory=list,
        description="각 질문의 중요도 점수 (0.0-1.0)"
    )

class ProposalValidation(BaseModel):
    """기획안 검증 결과"""
    feasibility_score: float = Field(description="실현 가능성 점수 (0.0-1.0)")
    market_fit_score: float = Field(description="시장 적합성 점수 (0.0-1.0)")
    innovation_score: float = Field(description="혁신성 점수 (0.0-1.0)")
    key_risks: List[str] = Field(description="주요 리스크 요인")
    recommendations: List[str] = Field(description="개선 권장사항")

# --- 2. 향상된 State 정의 ---
class ProposalGenerationState(TypedDict):
    original_proposal: str
    product_category: str
    category_confidence: float
    generated_questions: List[str]
    question_priorities: List[float]
    retrieved_docs: Dict[str, List[Document]]
    consolidated_context: str
    final_proposal: str
    validation_results: Optional[Dict[str, Any]]
    error_log: List[str]
    processing_time: Dict[str, float]

# --- 3. LLM 초기화 (Structured Output 지원) ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.7,
    max_retries=3
)

# Structured Output을 위한 LLM 설정
structured_llm = llm.with_structured_output

embedding_function = OpenAIEmbeddings()

# --- 4. Vector DB 및 Retriever 불러오기 ---
def initialize_retriever():
    """Retriever 초기화 및 에러 처리"""
    try:
        logger.info("🔄 기존 Vector DB와 Retriever 불러오는 중...")
        chroma_db_path = r'C:\cursor\langgraph_baseline\oliveyoung\chroma_db'
        vector_store = Chroma(
            collection_name="cosmetic_chunks", 
            persist_directory=chroma_db_path, 
            embedding_function=embedding_function
        )
        
        fs_path = Path(r"C:\cursor\langgraph_baseline\oliveyoung\data")
        store = LocalFileStore(fs_path)
        docstore = create_kv_docstore(store)
        
        retriever = ParentDocumentRetriever(
            vectorstore=vector_store, 
            docstore=docstore, 
            child_splitter=RecursiveCharacterTextSplitter(chunk_size=500)
        )
        logger.info("✅ Retriever 초기화 완료")
        return vector_store, retriever
    except Exception as e:
        logger.error(f"❌ Retriever 초기화 실패: {str(e)}")
        raise

vector_store, retriever = initialize_retriever()

# --- 5. 개선된 노드 함수들 ---

def extract_category_from_proposal(state: ProposalGenerationState) -> ProposalGenerationState:
    """Structured Output을 사용한 카테고리 분류"""
    start_time = datetime.now()
    logger.info("\n▶️ 단계 1: 제품 카테고리 분류 (Structured Output)")
    
    try:
        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 화장품 카테고리 분류 전문가입니다. 
            주어진 기획안을 분석하여 가장 적합한 카테고리를 선택하고 신뢰도를 평가해주세요.
            
            카테고리 목록: 스킨케어, 마스크팩, 클렌징, 선케어, 메이크업, 맨즈케어, 헤어케어, 바디케어, 향수/디퓨저"""),
            ("human", "기획안: {proposal}")
        ])
        
        chain = prompt | structured_llm(ProductCategory)
        result = chain.invoke({"proposal": state["original_proposal"]})
        
        logger.info(f"  - 분류된 카테고리: '{result.category}' (신뢰도: {result.confidence})")
        
        processing_time = (datetime.now() - start_time).total_seconds()
        return {
            **state, 
            "product_category": result.category,
            "category_confidence": result.confidence,
            "processing_time": {**state.get("processing_time", {}), "category_extraction": processing_time}
        }
    except Exception as e:
        logger.error(f"  - 카테고리 분류 실패: {str(e)}")
        return {
            **state,
            "product_category": "스킨케어",  # 기본값
            "category_confidence": 0.5,
            "error_log": state.get("error_log", []) + [f"카테고리 분류 오류: {str(e)}"]
        }

def generate_questions_from_proposal(state: ProposalGenerationState) -> ProposalGenerationState:
    """Structured Output을 사용한 질문 생성"""
    start_time = datetime.now()
    logger.info("\n▶️ 단계 2: 기획안 분석 및 질문 생성 (Structured Output)")
    
    try:
        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 시니어 화장품 PM(Product Manager)입니다.
            주어진 화장품 기획안을 분석하여, 레퍼런스 제품 정보를 검색할 수 있는 
            구체적이고 검색 가능한 질문들을 생성하세요.
            **가장 중요한 순서대로, 반드시 5개에서 7개 사이의 핵심 질문만 생성해주세요.**
            
            질문 작성 시 고려사항:
            1. 경쟁 제품 분석
               - "[카테고리] 제품 중 [핵심 성분/기능]을 포함한 제품들의 가격대와 리뷰는?"
               - "[타겟층]을 위한 [카테고리] 제품들의 주요 성분과 차별점은?"
            
            2. 가격 포지셔닝
               - "[유사 기능] [카테고리] 제품들의 가격 분포와 용량별 가격은?"
               - "[브랜드 포지셔닝]의 [카테고리] 제품 가격대는?"
             
            3. 성분 및 효능
               - "[핵심 성분]을 함유한 [카테고리] 제품들의 효능과 소비자 만족도는?"
               - "민감성 피부용 [카테고리]에서 주로 사용되는 진정 성분은?"
            
            4. 소비자 니즈
               - "[타겟층]이 [카테고리] 제품 선택 시 중요하게 고려하는 요소는?"
               - "[특정 피부 고민]을 가진 소비자들의 제품 사용 후기는?"
            
            5. 시장 트렌드
               - "최근 [카테고리] 시장의 주요 트렌드와 인기 제품은?"
            
            각 질문은 구체적이고 검색 가능한 키워드를 포함해야 합니다.
            **가장 중요한 순서대로, 반드시 5개에서 7개 사이의 핵심 질문만 생성해주세요.**
            """),
            ("human", "기획안: {proposal}\n카테고리: {category}")
        ])
        
        chain = prompt | structured_llm(ResearchQuestions)
        result = chain.invoke({
            "proposal": state["original_proposal"],
            "category": state["product_category"]
        })
        
        # 우선순위 자동 할당 (없을 경우)
        if not result.priority_scores:
            result.priority_scores = [1.0 - (i * 0.1) for i in range(len(result.questions))]
        
        logger.info(f"  - 생성된 질문 {len(result.questions)}개:")
        for i, (q, p) in enumerate(zip(result.questions, result.priority_scores)):
            logger.info(f"    {i+1}. [{p:.1f}] {q}")
        
        processing_time = (datetime.now() - start_time).total_seconds()
        return {
            **state,
            "generated_questions": result.questions,
            "question_priorities": result.priority_scores,
            "processing_time": {**state.get("processing_time", {}), "question_generation": processing_time}
        }
    except Exception as e:
        logger.error(f"  - 질문 생성 실패: {str(e)}")
        # 폴백 질문 제공 (더 구체적으로)
        fallback_questions = [
            f"{state['product_category']} 시장에서 3-5만원대 제품들의 주요 성분과 가격은?",
            f"민감성 피부용 {state['product_category']} 제품들의 진정 성분과 소비자 리뷰는?",
            f"시카, 판테놀, 나이아신아마이드를 함유한 {state['product_category']} 제품들의 효능과 가격은?",
            f"20-30대 여성이 선호하는 {state['product_category']} 제품의 특징과 구매 이유는?",
            f"인플루언서 협업 {state['product_category']} 제품들의 성공 요인과 가격 전략은?",
            f"광채, 톤업 기능을 가진 {state['product_category']} 제품들의 성분과 사용감 리뷰는?",
            f"젤크림 제형의 {state['product_category']} 제품들의 장단점과 가격대는?"
        ]
        return {
            **state,
            "generated_questions": fallback_questions[:7],  # 최대 7개
            "question_priorities": [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7][:7],
            "error_log": state.get("error_log", []) + [f"질문 생성 오류: {str(e)}"]
        }

async def execute_retrieval_for_questions(state: ProposalGenerationState) -> ProposalGenerationState:
    """개선된 병렬 검색 및 중복 제거"""
    start_time = datetime.now()
    logger.info(f"\n▶️ 단계 3: 질문별 병렬 검색 (카테고리: '{state['product_category']}')")
    
    questions = state["generated_questions"]
    priorities = state.get("question_priorities", [1.0] * len(questions))
    product_category = state["product_category"]
    
    # 캐시를 위한 문서 저장소
    doc_cache = {}
    
    async def search_for_question(question: str, priority: float):
        loop = asyncio.get_running_loop()
        
        try:
            # 우선순위에 따라 검색 수 조정
            k_value = int(30 + (priority * 20))  # 30-50 범위
            
            # 벡터 검색
            search_with_args = functools.partial(vector_store.similarity_search, k=k_value)
            child_docs = await loop.run_in_executor(None, search_with_args, question)
            
            # 카테고리 필터링 및 스코어링
            filtered_docs = []
            for doc in child_docs:
                doc_category = doc.metadata.get('category', '')
                if product_category in doc_category:
                    # 관련성 스코어 계산 (간단한 예시)
                    relevance_score = 1.0
                    if doc.metadata.get('brand', '').lower() in question.lower():
                        relevance_score += 0.2
                    if doc.metadata.get('name', '').lower() in question.lower():
                        relevance_score += 0.3
                    
                    doc.metadata['relevance_score'] = relevance_score
                    filtered_docs.append(doc)
            
            # 관련성 순으로 정렬
            filtered_docs.sort(key=lambda x: x.metadata.get('relevance_score', 0), reverse=True)
            filtered_docs = filtered_docs[:10]  # 상위 10개만
            
            # 부모 문서 추출 (중복 제거)
            parent_ids = list(set(doc.metadata.get(retriever.id_key) for doc in filtered_docs if retriever.id_key in doc.metadata))
            parent_docs = []
            
            for pid in parent_ids:
                if pid not in doc_cache:
                    docs = retriever.docstore.mget([pid])
                    if docs[0]:
                        doc_cache[pid] = docs[0]
                if pid in doc_cache:
                    parent_docs.append(doc_cache[pid])
            
            logger.info(f"  - '{question[:50]}...' → {len(parent_docs)}개 문서 검색됨")
            return question, parent_docs
            
        except Exception as e:
            logger.error(f"  - 검색 오류 ('{question[:30]}...'): {str(e)}")
            return question, []
    
    # 병렬 검색 실행
    tasks = [search_for_question(q, p) for q, p in zip(questions, priorities)]
    results = await asyncio.gather(*tasks)
    
    retrieved_docs = {question: docs for question, docs in results}
    
    # 검색 통계
    total_docs = sum(len(docs) for docs in retrieved_docs.values())
    logger.info(f"  - 총 {total_docs}개 고유 문서 검색 완료 (캐시 적중률: {len(doc_cache)}/{total_docs})")
    
    processing_time = (datetime.now() - start_time).total_seconds()
    return {
        **state,
        "retrieved_docs": retrieved_docs,
        "processing_time": {**state.get("processing_time", {}), "retrieval": processing_time}
    }

def consolidate_context(state: ProposalGenerationState) -> ProposalGenerationState:
    """개선된 컨텍스트 통합 (중복 제거 및 구조화)"""
    start_time = datetime.now()
    logger.info("\n▶️ 단계 4: 검색 결과 통합 및 구조화")
    
    try:
        retrieved_docs = state["retrieved_docs"]
        priorities = state.get("question_priorities", [1.0] * len(state["generated_questions"]))
        
        # 문서별 중요도 계산
        doc_importance = defaultdict(float)
        doc_questions = defaultdict(list)
        
        # priorities와 questions의 길이가 맞지 않을 경우 처리
        questions_list = list(retrieved_docs.keys())
        if len(priorities) < len(questions_list):
            priorities.extend([0.5] * (len(questions_list) - len(priorities)))
        
        for question, priority in zip(questions_list, priorities):
            docs = retrieved_docs.get(question, [])
            for doc in docs:
                doc_id = str(doc.metadata.get('source', '')) + str(doc.metadata.get('name', ''))
                doc_importance[doc_id] += priority
                doc_questions[doc_id].append(question)
        
        # 구조화된 컨텍스트 생성
        context_sections = {
            "핵심_제품_정보": [],
            "시장_트렌드": [],
            "경쟁_제품_분석": [],
            "고객_리뷰_인사이트": [],
            "성분_및_효능": []
        }
        
        processed_docs = set()
        
        for question, docs in retrieved_docs.items():
            for doc in docs:
                doc_id = str(doc.metadata.get('source', '')) + str(doc.metadata.get('name', ''))
                
                if doc_id in processed_docs:
                    continue
                processed_docs.add(doc_id)
                
                meta = doc.metadata
                doc_info = {
                    "제품명": meta.get('name', '제품명 정보 없음'),
                    "브랜드": meta.get('brand', '브랜드 정보 없음'),
                    "가격": str(meta.get('price', '가격 정보 없음')),
                    "용량": str(meta.get('volume', '용량 정보 없음')),
                    "주요_내용": doc.page_content if doc.page_content else "",
                    "관련_질문": doc_questions.get(doc_id, []),
                    "중요도": doc_importance.get(doc_id, 0)
                }
                
                # 섹션별 분류 (None 체크 추가)
                review_text = str(meta.get('review', ''))
                ingredients_text = str(meta.get('ingredients', ''))
                page_content = doc.page_content.lower() if doc.page_content else ""
                
                if review_text and "리뷰" in review_text:
                    context_sections["고객_리뷰_인사이트"].append(doc_info)
                elif ingredients_text and "성분" in ingredients_text:
                    context_sections["성분_및_효능"].append(doc_info)
                elif any(keyword in page_content for keyword in ["트렌드", "시장", "성장"]):
                    context_sections["시장_트렌드"].append(doc_info)
                elif meta.get('brand', '') and meta.get('brand', '') != state.get('brand', ''):
                    context_sections["경쟁_제품_분석"].append(doc_info)
                else:
                    context_sections["핵심_제품_정보"].append(doc_info)
        
        # 각 섹션 정렬 (중요도 순)
        for section in context_sections.values():
            section.sort(key=lambda x: x["중요도"], reverse=True)
        
        # 최종 컨텍스트 생성
        consolidated_parts = []
        for section_name, docs in context_sections.items():
            if docs:
                consolidated_parts.append(f"\n=== {section_name.replace('_', ' ')} ===")
                for i, doc in enumerate(docs[:5], 1):  # 각 섹션당 최대 5개
                    # 메타데이터에서 상세 정보 추출
                    meta = next((d.metadata for d in retrieved_docs.get(doc['관련_질문'][0], []) 
                                if d.metadata.get('name') == doc['제품명']), {})
                    
                    consolidated_parts.append(f"\n[{i}] {doc['브랜드']} - {doc['제품명']}")
                    consolidated_parts.append(f"가격/용량: {doc['가격']}원 / {doc['용량']}")
                    
                    # 추가 메타데이터 정보
                    if meta:
                        if meta.get('manufacturer'):
                            consolidated_parts.append(f"제조/판매: {meta.get('manufacturer')}")
                        if meta.get('ingredients'):
                            consolidated_parts.append(f"주요 성분: {meta.get('ingredients')}...")
                        if meta.get('review'):
                            consolidated_parts.append(f"리뷰 요약: {meta.get('review')}...")
                    
                    consolidated_parts.append(f"내용: {doc['주요_내용']}")
                    consolidated_parts.append("-" * 50)
        
        # 전체 검색 결과 요약 추가
        consolidated_parts.insert(0, f"\n📊 검색 결과 요약")
        consolidated_parts.insert(1, f"- 총 {len(processed_docs)}개 제품 검색")
        consolidated_parts.insert(2, f"- 카테고리: {state['product_category']}")
        consolidated_parts.insert(3, f"- 주요 가격대: 분석 필요")
        consolidated_parts.insert(4, f"- 주요 성분 트렌드: 분석 필요")
        consolidated_parts.insert(5, "=" * 50)
        
        consolidated_context = "\n".join(consolidated_parts)
        
        logger.info(f"  - {len(processed_docs)}개 고유 문서를 {len([s for s in context_sections.values() if s])}개 섹션으로 구조화")
        
        processing_time = (datetime.now() - start_time).total_seconds()
        return {
            **state,
            "consolidated_context": consolidated_context,
            "processing_time": {**state.get("processing_time", {}), "consolidation": processing_time}
        }
    except Exception as e:
        logger.error(f"  - 컨텍스트 통합 실패: {str(e)}")
        logger.error(f"  - 오류 세부사항: {type(e).__name__}")
        import traceback
        logger.error(traceback.format_exc())
        
        # 기본 컨텍스트 생성
        fallback_context = "검색된 문서를 처리하는 중 오류가 발생했습니다."
        return {
            **state,
            "consolidated_context": fallback_context,
            "error_log": state.get("error_log", []) + [f"컨텍스트 통합 오류: {str(e)}"],
            "processing_time": {**state.get("processing_time", {}), "consolidation": 0.0}
        }

def validate_proposal(state: ProposalGenerationState) -> ProposalGenerationState:
    """기획안 검증 단계 (새로운 노드)"""
    start_time = datetime.now()
    logger.info("\n▶️ 단계 5: 기획안 타당성 검증")
    
    try:
        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 화장품 업계의 전문 컨설턴트입니다.
            주어진 기획안과 리서치 자료를 바탕으로 타당성을 평가해주세요."""),
            ("human", """
            기획안: {proposal}
            카테고리: {category}
            리서치 자료: {context}
            
            위 정보를 바탕으로 기획안의 타당성을 평가해주세요.
            """)
        ])
        
        chain = prompt | structured_llm(ProposalValidation)
        result = chain.invoke({
            "proposal": state["original_proposal"],
            "category": state["product_category"],
            "context": state["consolidated_context"][:2000]  # 토큰 제한
        })
        
        validation_dict = {
            "feasibility_score": result.feasibility_score,
            "market_fit_score": result.market_fit_score,
            "innovation_score": result.innovation_score,
            "key_risks": result.key_risks,
            "recommendations": result.recommendations
        }
        
        logger.info(f"  - 타당성 점수: {result.feasibility_score:.2f}")
        logger.info(f"  - 시장 적합성: {result.market_fit_score:.2f}")
        logger.info(f"  - 혁신성: {result.innovation_score:.2f}")
        
        processing_time = (datetime.now() - start_time).total_seconds()
        return {
            **state,
            "validation_results": validation_dict,
            "processing_time": {**state.get("processing_time", {}), "validation": processing_time}
        }
    except Exception as e:
        logger.error(f"  - 검증 실패: {str(e)}")
        return {
            **state,
            "validation_results": None,
            "error_log": state.get("error_log", []) + [f"검증 오류: {str(e)}"]
        }

def generate_final_proposal(state: ProposalGenerationState) -> ProposalGenerationState:
    """최종 기획안 생성 (검증 결과 반영)"""
    start_time = datetime.now()
    logger.info("\n▶️ 단계 6: 최종 기획안 생성")
    
    validation = state.get("validation_results", {})
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 국내 최고의 화장품 상품기획 전문가입니다.
        주어진 기획 초안과 리서치 자료를 바탕으로, 완성도 높은 최종 기획안을 작성해주세요.
        
        리서치 자료에는 실제 제품들의 정보가 포함되어 있으며, 각 제품의 메타데이터를 참고하여:
        - 경쟁 제품 분석
        - 가격 포지셔닝 전략
        - 성분 벤치마킹
        - 소비자 리뷰 인사이트
        등을 구체적으로 반영해주세요.
        
        최종 기획안은 다음 구조를 반드시 따라주세요:
        
        ### 1. 제품 컨셉 (핵심 스토리)
        - 한 줄 컨셉
        - 컨셉 스토리 (1-2문장)
        
        ### 2. 맞춤형 컨셉 설명
        - 유튜버/인플루언서의 철학 반영 (해당시)
        - 구독자/타겟 고객 니즈 충족
        - 연결고리 서술
        
        ### 3. 타겟 페르소나
        - 인구통계학적 정보
        - 피부 타입 및 고민
        - 핵심 Pain Point (3가지)
        - 핵심 소구점/Unmet Needs (3가지)
        
        ### 4. 제품 스펙 (컨셉 구현 방안)
        - 제형 (경쟁 제품 분석 기반)
        - 핵심 성분 (리서치된 제품들의 성분 참고)
        - 기능 (4가지 주요 기능)
        - 사용법
        - 패키징
        - 차별화 포인트 (3가지)
        
        ### 5. 경쟁 제품 분석 및 차별화 전략
        - 주요 경쟁 제품 리스트 (리서치 자료에서 5개 이상 선정)
        - 각 제품의 장단점 분석 (가격, 성분, 리뷰 기반)
        - 시장의 미충족 니즈
        - 우리 제품의 지속 가능한 경쟁 우위
        
        ### 6. 마케팅 및 커뮤니케이션 전략
        - 핵심 메시지
        - 기대치 관리 및 신뢰도 구축
        - 타겟별 메시지 구체화
        
        리서치 자료의 제품 정보를 인용할 때는 반드시 다음과 같은 형식으로 명시해주세요:
        예시: "제품명(가격대, 성분)의 경우..."
        
        특히 가격 설정시에는 검색된 유사 제품들의 가격대를 구체적으로 언급하며 근거를 제시해주세요.
        """),
        ("human", """
        **[기획 초안]**
        {original_proposal}
        
        **[카테고리]**
        {category} (신뢰도: {confidence})
        
        **[검색된 레퍼런스 제품 및 시장 리서치 자료]**
        {consolidated_context}
        
        **[타당성 검증 결과]**
        - 실현 가능성: {feasibility}
        - 시장 적합성: {market_fit}
        - 혁신성: {innovation}
        - 주요 리스크: {risks}
        - 개선 권장사항: {recommendations}
        
        위 정보를 모두 활용하여, 구조화된 최종 기획안을 작성해주세요.
        특히 리서치 자료의 실제 제품 정보(브랜드, 제품명, 가격, 성분, 리뷰)를 적극 활용하여 
        구체적이고 실현 가능한 기획안을 작성해주세요. 
        - **타겟군(여성)**에 맞춰서 레퍼런스를 참조하고, 타겟군의 니즈를 충족하는 기획안을 작성해주세요.
        - **남성용 제품은 절대 참고하지 마세요.**
        """)
    ])
    
    chain = prompt | llm
    response = chain.invoke({
        "original_proposal": state["original_proposal"],
        "category": state["product_category"],
        "confidence": state.get("category_confidence", 1.0),
        "consolidated_context": state["consolidated_context"],
        "feasibility": validation.get("feasibility_score", "미평가"),
        "market_fit": validation.get("market_fit_score", "미평가"),
        "innovation": validation.get("innovation_score", "미평가"),
        "risks": ", ".join(validation.get("key_risks", ["미평가"])),
        "recommendations": ", ".join(validation.get("recommendations", ["미평가"]))
    })
    
    processing_time = (datetime.now() - start_time).total_seconds()
    total_time = sum(state.get("processing_time", {}).values()) + processing_time
    
    logger.info(f"  - 최종 기획안 생성 완료!")
    logger.info(f"  - 총 처리 시간: {total_time:.2f}초")
    
    return {
        **state,
        "final_proposal": response.content,
        "processing_time": {**state.get("processing_time", {}), "final_generation": processing_time}
    }

# --- 6. 개선된 그래프 생성 ---
def create_full_generation_graph():
    """향상된 워크플로우 그래프 생성"""
    graph = StateGraph(ProposalGenerationState)
    
    # 노드 추가
    graph.add_node("extract_category", extract_category_from_proposal)
    graph.add_node("generate_questions", generate_questions_from_proposal)
    graph.add_node("execute_search", execute_retrieval_for_questions)
    graph.add_node("consolidate_context", consolidate_context)
    graph.add_node("validate_proposal", validate_proposal)
    graph.add_node("generate_final", generate_final_proposal)
    
    # 엣지 설정
    graph.set_entry_point("extract_category")
    graph.add_edge("extract_category", "generate_questions")
    graph.add_edge("generate_questions", "execute_search")
    graph.add_edge("execute_search", "consolidate_context")
    graph.add_edge("consolidate_context", "validate_proposal")
    graph.add_edge("validate_proposal", "generate_final")
    graph.add_edge("generate_final", END)
    
    return graph.compile()

# --- 7. 메인 실행 함수 ---
async def main(proposal_text: str):
    """메인 실행 함수"""
    app = create_full_generation_graph()
    
    # 초기 상태 설정
    initial_state: ProposalGenerationState = {
        "original_proposal": proposal_text,
        "product_category": "",
        "category_confidence": 0.0,
        "generated_questions": [],
        "question_priorities": [],
        "retrieved_docs": {},
        "consolidated_context": "",
        "final_proposal": "",
        "validation_results": None,
        "error_log": [],
        "processing_time": {}
    }
    
    print("\n" + "="*80)
    print("🚀 화장품 기획안 생성 프로세스 시작 (Enhanced Version)")
    print("="*80)
    
    try:
        # 그래프 실행
        final_state = None
        async for update in app.astream(initial_state):
            node_name = list(update.keys())[0]
            final_state = update[node_name]
        
        # 결과 출력
        print("\n" + "="*80)
        print("✅ 전체 프로세스 완료")
        print("="*80)
        
        if final_state:
            # 처리 시간 요약
            print("\n📊 처리 시간 분석:")
            for step, time in final_state.get("processing_time", {}).items():
                print(f"  - {step}: {time:.2f}초")
            print(f"  - 총 시간: {sum(final_state.get('processing_time', {}).values()):.2f}초")
            
            # 검증 결과 요약
            if final_state.get("validation_results"):
                print("\n📈 타당성 검증 결과:")
                val = final_state["validation_results"]
                print(f"  - 실현 가능성: {val.get('feasibility_score', 0):.2f}/1.0")
                print(f"  - 시장 적합성: {val.get('market_fit_score', 0):.2f}/1.0")
                print(f"  - 혁신성: {val.get('innovation_score', 0):.2f}/1.0")
            
            # 오류 로그
            if final_state.get("error_log"):
                print("\n⚠️ 발생한 오류:")
                for error in final_state["error_log"]:
                    print(f"  - {error}")
            
            # 최종 기획안 출력
            print("\n" + "="*80)
            print("📝 완성된 최종 화장품 기획안")
            print("="*80)
            print(final_state.get("final_proposal", "기획안 생성에 실패했습니다."))
            
    except Exception as e:
        logger.error(f"프로세스 실행 중 오류 발생: {str(e)}")
        print(f"\n❌ 오류 발생: {str(e)}")

2025-07-23 12:40:10,618 - INFO - 🔄 기존 Vector DB와 Retriever 불러오는 중...
2025-07-23 12:40:10,697 - INFO - ✅ Retriever 초기화 완료


In [38]:
sample_proposal = """
### 1. 제품 컨셉 (핵심 스토리)

**"세영수퍼의 '개복치도 갓생 사는 ✨데일리 글로우✨' 철학을 담은 저자극 스킨케어 톤업 베이스로, 민감성 피부도 안심하고 매일매일 맑고 생기 넘치는 피부 자신감을 선사하는 데일리 필수템."**

이 제품은 매일 바빠도 피부에 건강한 생기를 더하고 싶고, 피부가 예민해도 마음껏 아름다움을 뽐내고 싶은 당신의 '갓생'을 응원합니다. "우리 개복치들은 이거 하나면 쌩얼 자신감 뿜뿜!"

### 2. 맞춤형 컨셉 설명

이 제품은 '세영수퍼'님 스스로를 **'개복치 피부'**라 칭하며 민감성에 대한 깊은 공감을 표하는 것에서 출발합니다. 그녀의 솔직하고 유머러스한 '찐친' 같은 채널 톤앤매너를 담아, 민감성 피부로 인해 베이스 메이크업에 어려움을 겪는 구독자들이 **'안심하고 갓생 살듯' 매일매일 사용할 수 있도록** 기획되었습니다. '꾸안꾸' 스타일을 선호하며 자연스러운 피부 표현에 대한 니즈가 높은 팬덤에게, 답답함 없이 가볍고 촉촉하게 밀착되면서도 칙칙함 없이 맑고 생기 있는 '데일리 글로우'를 선사하는 맞춤형 제품입니다. 특히 그녀가 강조하는 "피부가 왜 이렇게 좋냐고 다들 물어보시더라고", "얼굴이 밝아졌다는 소리 많이 들었습니다"와 같은 자신감 넘치는 피부를 누구나 쉽게 연출할 수 있도록 돕는 데 중점을 둡니다.

### 3. 선정 근거 (각 근거에 쇼츠/롱폼 구분 명시, 최근 6개월 데이터 우선)

- **데이터 근거 1: 인플루언서의 '개복치 피부' 정체성과 스킨케어 선호도**
    - **유튜버 페르소나:** 세영수퍼님은 스스로를 **"개복치 피부"** (V1 롱폼 '클렌징밀크 폼클렌징', V15 롱폼 '피부좋아지는법')라고 칭하며, 민감성 피부임을 명확히 언급합니다. 이로 인해 그녀가 추천하는 제품들은 자연스럽게 '순하고 자극 없는' 특성을 지니게 됩니다.
    - **선호하는 스킨케어 제형:** "무거운 크림 보다는 가벼운 제품을 바꾸니까 훨씬 효과를 많이 봤어" (V15 롱폼 '피부좋아지는법'), "묵직한 제형보다는 가벼운 제품" (V15 롱폼)이라는 언급에서 알 수 있듯, 답답함 없이 가볍고 촉촉한 제형을 선호합니다. 또한, "순수 비타민이 들어가있어서 믿을만 했고 또 그래서 효과도 본 것 같아요" (V10 롱폼 '진짜나만알고싶었는데')라고 말해 성분과 효과를 중요하게 여깁니다.
- **데이터 근거 2: 팬덤의 피부 고민 및 '워너비' 피부 표현 니즈**
    - **민감성 피부 공감대:** "저는 OO피부인데 괜찮을까요?" (V15 롱폼 스크립트)와 같이 구독자들 또한 개인의 피부 고민, 특히 민감성 또는 트러블성 피부에 대한 질문이 많아 '안심하고 쓸 수 있는' 제품에 대한 갈증이 높습니다.
    - **자연스럽고 깨끗한 피부 표현 워너비:** "쌩얼도 넘 예쁘신데 메컵 후 넘넘 예뿌심" (V30 쇼츠 '회사에서 몰래 화장하기')과 같이 구독자들은 세영수퍼님의 깨끗하고 건강한 피부 표현에 대한 칭찬과 동경을 드러냅니다. "요철 가리는 제품 뭔지 알 수 있을까요?!" (V20 롱폼 '애견녀 메이크업'), "파우더 정보 부탁해요ㅠㅠ" (V6 롱폼 '내가제일조아하는연한화장') 등의 질문은 단순히 커버를 넘어 매끈하고 자연스러운 피부결 연출에 대한 높은 니즈를 보여줍니다.
    - **'톤업/광채'에 대한 관심:** V10 롱폼에서 "피부톤이 맑아지고 화사하게 케어"되는 비타민 C 크림을 추천하고, V29 롱폼에서 블러셔 사용 후 "광택이 좌르르르"한 효과를 강조하는 점은 팬덤 역시 칙칙함 없이 맑고 생기 있는 피부 표현에 관심이 많음을 시사합니다.
- **데이터 근거 3: 채널 고유 문화와 제품 컨셉의 시너지**
    - **'현실 밀착 꾸안꾸'와 '갓생' 지향:** 세영수퍼님은 "연한화장 조회수가 잘나온다는건.. 니즈가 확실히 있다" (V6 롱폼)고 본인도 인지하듯, 일상에서 쉽게 적용 가능한 '꾸안꾸' 메이크업 팁을 주로 제공합니다. '애견녀 메이크업' (V20 롱폼), '회사에서 몰래 화장하기' (V28 쇼츠) 등 일상생활 속 '갓생'을 지향하는 팬덤의 라이프스타일에 부합하는, 간편하면서도 효과적인 제품에 대한 수요가 높습니다.
    - **'찐텐' 사용법과 솔직한 추천에 대한 신뢰:** 'shorts' 영상이 많아 빠르고 직관적인 팁을 제공하며, "이거 알려줘도 되나?" (V1 롱폼), "진짜 나만 알고 싶었는데" (V10 롱폼)와 같이 친구에게 알려주듯 솔직하고 친근한 말투로 정보를 전달합니다. 팬덤은 "립 궁금해요!", "쿠션 뭐에요?" (V9, V13, V20, V23, V24, V26, V28, V29, V30, V32, V34 등 반복적으로 등장)와 같이 유튜버의 사용 제품 정보를 적극적으로 요청하며, 그녀의 추천에 대한 높은 신뢰도를 보여줍니다. 이는 제품 출시 시 유튜버의 '찐텐' 사용법 강조 마케팅이 강력한 시너지를 낼 수 있음을 의미합니다.

### 4. 제품 스펙 (컨셉 구현 방안)

- **카테고리:** 스킨케어 톤업 베이스 (Skin-Care Tone-Up Base)
    - *서브 카테고리:* 진정 케어, 내추럴 글로우 톤업 크림 (또는 밤/로션 제형)
- **타겟 페르소나:**
'세영수퍼'님처럼 솔직하고 당당한 매력과 더불어 '개복치 피부'를 가진 자신의 특성을 이해하고, 예민한 피부에도 부담 없이 '맑고 생기 있는 꾸안꾸' 피부 표현을 연출하고 싶은 20-30대 여성 팬덤.
- **핵심 특성:**
    1. **① 개복치 안심 진정 포뮬러:**
        - 피부 진정에 탁월한 시카, 병풀 추출물, 판테놀, 마데카소사이드 등 핵심 스킨케어 성분을 고함량 함유하여 민감성 피부도 안심하고 사용할 수 있는 저자극 포뮬러. (민감성 피부 자극 테스트 완료, 20가지 주의 성분 배제)
        - "저는 예민한 피부다", "크림고민중이였는대 감사해요" (V15 롱폼), "미백템은 사실 얼굴이 잘 뒤집어져서 조심스러운 경향이 있거든요" (V10 롱폼) 등 유튜버와 팬덤의 민감성 피부 고민을 해결.
    2. **② 내추럴 톤업 & 매끈 결광:**
        - 본연의 피부 톤을 자연스럽게 화사하게 밝혀주고, 미세한 요철을 부드럽게 커버하여 타고난 듯 매끈하고 맑은 결광 피부를 연출. (세영수퍼님이 선호하는 '가볍고 촉촉한 제형' 구현).
        - "얼굴이 밝아졌다는 소리 많이 들었습니다" (V10 롱폼), "요철 가리는 제품 뭔지 알 수 있을까요?!" (V20 롱폼) 등 팬덤의 니즈 충족.
    3. **③ 갓생 지속력 & '멀쩡'한 마무리감:**
        - 가볍게 밀착되어 피부에 답답함 없이 오랜 시간 '멀쩡하게' 지속되며, 마스크나 외부 자극에도 쉽게 무너지지 않아 '갓생' 라이프스타일 속 자신감을 유지.
        - 세영수퍼님이 '더운 여름에도 멀쩡한' 메이크업을 언급하며 지속력을 강조한 점(V4 롱폼)을 반영.
    4. **④ '똥손'도 쉬운 블렌딩 & 활용:**
        - 손가락이나 스펀지로 쉽게 블렌딩하여 경계 없이 자연스러운 피부 표현이 가능하며, 파데프리 데일리 톤업, 메이크업 부스팅 베이스 등 다양하게 활용 가능.
        - "똥손" (V2 롱폼 댓글)도 쉽게 따라할 수 있는 실용적인 사용성을 중시하는 채널의 특성 반영.
"""
        # 비동기 실행
await main(sample_proposal)

2025-07-23 12:55:38,052 - INFO - 
▶️ 단계 1: 제품 카테고리 분류 (Structured Output)



🚀 화장품 기획안 생성 프로세스 시작 (Enhanced Version)


2025-07-23 12:55:47,555 - INFO -   - 분류된 카테고리: '스킨케어' (신뢰도: 0.95)
2025-07-23 12:55:47,557 - INFO - 
▶️ 단계 2: 기획안 분석 및 질문 생성 (Structured Output)
2025-07-23 12:56:01,861 - INFO -   - 생성된 질문 6개:
2025-07-23 12:56:01,862 - INFO -     1. [0.9] 민감성 피부용 스킨케어 톤업 베이스 시장 분석: 경쟁 제품 가격대, 핵심 성분(시카, 병풀, 판테놀), 제형(크림/로션/밤), 주요 효능(진정, 톤업, 결광) 및 소비자 리뷰 (장단점)는?
2025-07-23 12:56:01,862 - INFO -     2. [0.9] 20-30대 여성 타겟의 '꾸안꾸', '내추럴 글로우' 컨셉 저자극 톤업 베이스 제품들이 강조하는 차별점과 마케팅 메시지는?
2025-07-23 12:56:01,863 - INFO -     3. [0.8] 스킨케어 기능(진정, 보습)을 강조한 톤업 베이스 제품들의 용량별 가격 분포 및 프리미엄/매스 포지셔닝 전략은?
2025-07-23 12:56:01,864 - INFO -     4. [0.8] 민감성 피부 소비자들이 톤업 베이스 선택 시 가장 중요하게 고려하는 요소(성분 안정성, 발림성, 지속력, 마무리감, 톤업 효과)와 주요 불만족 사항은?
2025-07-23 12:56:01,865 - INFO -     5. [0.8] 최근 1년 이내 출시된 '스킨케어 톤업 베이스' 신제품 중 인기 있는 제품들의 공통점(성분, 제형, 컨셉)과 시장 트렌드 변화는?
2025-07-23 12:56:01,865 - INFO -     6. [0.7] '개복치 피부' 또는 '예민 피부' 키워드로 언급되는 뷰티 커뮤니티/리뷰에서 톤업 베이스 관련 피부 트러블 유발 사례 및 해결 방안은?
2025-07-23 12:56:01,867 - INFO - 
▶️ 단계 3: 질문별 병렬 검색 (카테고리:


✅ 전체 프로세스 완료

📊 처리 시간 분석:
  - category_extraction: 9.50초
  - question_generation: 14.31초
  - retrieval: 1.25초
  - consolidation: 0.00초
  - validation: 10.56초
  - final_generation: 51.64초
  - 총 시간: 87.27초

📈 타당성 검증 결과:
  - 실현 가능성: 0.90/1.0
  - 시장 적합성: 0.95/1.0
  - 혁신성: 0.80/1.0

📝 완성된 최종 화장품 기획안
국내 최고의 화장품 상품기획 전문가로서, 세영수퍼님의 '개복치도 갓생 사는 데일리 글로우' 철학을 담은 최종 기획안을 아래와 같이 제안합니다.

---

### 최종 기획안: 세영수퍼 시그니처 '개복치도 갓생 사는 데일리 글로우 톤업 베이스'

### 1. 제품 컨셉 (핵심 스토리)

**"세영수퍼의 '개복치도 갓생 사는 ✨데일리 글로우✨' 철학을 담은 저자극 스킨케어 톤업 베이스로, 민감성 피부도 안심하고 매일매일 맑고 생기 넘치는 피부 자신감을 선사하는 데일리 필수템."**

이 제품은 매일 바빠도 피부에 건강한 생기를 더하고 싶고, 피부가 예민해도 마음껏 아름다움을 뽐내고 싶은 당신의 '갓생'을 응원합니다. "우리 개복치들은 이거 하나면 쌩얼 자신감 뿜뿜!"

### 2. 맞춤형 컨셉 설명

이 제품은 '세영수퍼'님 스스로를 **'개복치 피부'**라 칭하며 민감성에 대한 깊은 공감을 표하는 것에서 출발합니다. 그녀의 솔직하고 유머러스한 '찐친' 같은 채널 톤앤매너를 담아, 민감성 피부로 인해 베이스 메이크업에 어려움을 겪는 구독자들이 **'안심하고 갓생 살듯' 매일매일 사용할 수 있도록** 기획되었습니다. '꾸안꾸' 스타일을 선호하며 자연스러운 피부 표현에 대한 니즈가 높은 팬덤에게, 답답함 없이 가볍고 촉촉하게 밀착되면서도 칙칙함 없이 맑고 생기 있는 '데일리 글로우'를 선사하는 맞춤형 제품입니다. 특히 그녀가 강조하는 "피부가 왜